# Replicate the length-three convergence figure

This notebook rebuilds Figure 2 of the manuscript — the length-three convergence figure — from the parquet deposit. The figure makes a structural argument that a forest plot cannot: that the **systematic scan surfaces convergence on a small set of deprivation-patterned endpoints**, not just isolated large IRRs.

Two panels:

1. **Panel A** — nine selected length-three deprivation contrasts, grouped by third-condition endpoint (COPD, drug/alcohol, heart failure, depression).
2. **Panel B** — the endpoint frequency in the top-60 deprivation-ranked length-three candidate set, demonstrating that the convergence in Panel A is a property of the ranked pool, not the curated rows.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import incigraph as ig
from incigraph.ci import irr_ci

ig.set_data_dir('./incigraph_data')  # update to wherever your parquet lives
print(f'incigraph version {ig.__version__}')

## Panel A: nine curated length-three contrasts

The nine rows shown were selected from the systematic scan output (see `step3_contrast_scan.py` and `curate_ordered_transition_candidates.py --sequence-length 3`). For each row we re-verify the IRR by re-computing it from the parquet, so the notebook serves as a reproducibility receipt.

In [ ]:
# Each row: (endpoint, label, sequence, stratification, comparison, reference)
# The endpoint key drives the grouping and colour; the rest describes the
# parquet query that recovers the IRR.
PANEL_A_SPECS = [
    # --- COPD endpoint ---
    dict(endpoint='COPD',
         label='Hypertension → Asthma → COPD',
         sublabel='IMD 5 vs IMD 1 — within White',
         sequence=[3, 23, 24], stratification='ETHNICITY+IMD',
         comparison={'ethnicity': 'WHITE', 'imd': 5.0},
         reference={'ethnicity': 'WHITE', 'imd': 1.0}),
    dict(endpoint='COPD',
         label='Osteoarthritis → Asthma → COPD',
         sublabel='IMD 5 vs IMD 1 — within White',
         sequence=[30, 23, 24], stratification='ETHNICITY+IMD',
         comparison={'ethnicity': 'WHITE', 'imd': 5.0},
         reference={'ethnicity': 'WHITE', 'imd': 1.0}),
    dict(endpoint='COPD',
         label='Depression → Anxiety → COPD',
         sublabel='IMD 5 vs IMD 1 — within Female',
         sequence=[10, 11, 24], stratification='IMD+SEX',
         comparison={'sex': 'F', 'imd': 5.0},
         reference={'sex': 'F', 'imd': 1.0}),
    # --- Drug/alcohol endpoint ---
    dict(endpoint='DA',
         label='Hypertension → Type 2 diabetes → Drug/alcohol misuse',
         sublabel='IMD 5 vs IMD 1 — unconditional',
         sequence=[3, 8, 60], stratification='IMD',
         comparison={'imd': 5.0}, reference={'imd': 1.0}),
    dict(endpoint='DA',
         label='Osteoarthritis → Hypertension → Drug/alcohol misuse',
         sublabel='IMD 4+5 vs IMD 1+2 — within White',
         sequence=[30, 3, 60], stratification='ETHNICITY+IMD',
         pooled_imd=True, conditioning={'ethnicity': 'WHITE'}),
    # --- Heart failure endpoint ---
    dict(endpoint='HF',
         label='Valvular disease → AF → Heart failure',
         sublabel='IMD 4+5 vs IMD 1+2 — unconditional',
         sequence=[5, 2, 1], stratification='IMD',
         pooled_imd=True, conditioning={}),
    dict(endpoint='HF',
         label='Osteoarthritis → AF → Heart failure',
         sublabel='IMD 4+5 vs IMD 1+2 — within White',
         sequence=[30, 2, 1], stratification='ETHNICITY+IMD',
         pooled_imd=True, conditioning={'ethnicity': 'WHITE'}),
    # --- Depression endpoint ---
    dict(endpoint='DEP',
         label='Cancer → Anxiety → Depression',
         sublabel='IMD 5 vs IMD 1 — within Male',
         sequence=[22, 11, 10], stratification='IMD+SEX',
         comparison={'sex': 'M', 'imd': 5.0},
         reference={'sex': 'M', 'imd': 1.0}),
    dict(endpoint='DEP',
         label='Epilepsy → Anxiety → Depression',
         sublabel='IMD 5 vs IMD 1 — unconditional',
         sequence=[21, 11, 10], stratification='IMD',
         comparison={'imd': 5.0}, reference={'imd': 1.0}),
]

In [ ]:
# Helper for pooled-IMD comparisons (4+5 vs 1+2). The API's compute_irr
# expects single-row filters; for pooled comparisons we sum the two rows
# and call irr_ci directly.
def compute_pooled_imd(spec):
    df = ig.get_sequence(spec['sequence'], stratification=spec['stratification'])
    df = df[~df['imd_missing']]
    # apply any conditioning (e.g. within White)
    for col, val in spec.get('conditioning', {}).items():
        df = df[df[col].astype(str).str.upper() == str(val).upper()]
    c = df[df['imd'].isin([4.0, 5.0])][['numerator', 'denominator']].sum()
    r = df[df['imd'].isin([1.0, 2.0])][['numerator', 'denominator']].sum()
    return irr_ci(c['numerator'], c['denominator'],
                  r['numerator'], r['denominator'])

rows = []
for spec in PANEL_A_SPECS:
    if spec.get('pooled_imd'):
        r = compute_pooled_imd(spec)
    else:
        df = ig.get_sequence(spec['sequence'], stratification=spec['stratification'])
        r = ig.compute_irr(df, comparison=spec['comparison'],
                           reference=spec['reference'])
    rows.append({
        'endpoint': spec['endpoint'],
        'label':    spec['label'],
        'sublabel': spec['sublabel'],
        'irr':      r['irr'],
        'lo':       r['lower_ci'],
        'hi':       r['upper_ci'],
        'n_comp':   int(r['n_comp']),
        'n_ref':    int(r['n_ref']),
    })
panel_A = pd.DataFrame(rows)
panel_A[['endpoint', 'label', 'irr', 'lo', 'hi', 'n_comp', 'n_ref']]

## Panel B: third-condition concentration

Panel B comes from the top-60 deprivation-ranked length-three candidate set produced by `curate_ordered_transition_candidates.py --sequence-length 3 --min-events 30 --top-n 60`. We re-compute that ranking inline here so the notebook is fully self-contained.

In [ ]:
# Reload the length-3 estimates and run a simple version of the contrast
# scan restricted to deprivation contrasts -- enough to identify the
# top-60 and check the third-condition concentration.
L3 = ig.load_estimates(3)
print(f'Length-3 estimates: {len(L3):,} rows')

# For each (sequence, stratification) compute the IMD 5 vs IMD 1 contrast
# in every conditioning stratum where both groups have >= 30 events. The
# step3 script does this for every contrast type; we narrow to deprivation.
from incigraph.disease_index import DISEASE_NAMES

MIN_EVENTS = 30
candidates = []
imd_strats = [s for s in ig.available_stratifications() if 'IMD' in s]
for strat in imd_strats:
    block = L3[L3['stratification_key'].astype(str) == strat]
    if block.empty: continue
    # other axes besides IMD
    other_cols = [
        {'AGE_CATG': 'age_catg', 'ETHNICITY': 'ethnicity', 'SEX': 'sex'}[a]
        for a in strat.split('+') if a != 'IMD'
    ]
    block = block[~block['imd_missing']]
    if other_cols:
        # drop rows where any conditioning value is missing
        for c in other_cols:
            block = block[block[c].notna()]
    # group within (sequence, conditioning), pick IMD=5 and IMD=1
    group_cols = ['sequence', 'target_disease_idx',
                  'target_disease_short'] + other_cols
    for keys, sub in block.groupby(group_cols, observed=True):
        n_c = sub[sub['imd'] == 5.0][['numerator', 'denominator']].sum()
        n_r = sub[sub['imd'] == 1.0][['numerator', 'denominator']].sum()
        if min(n_c['numerator'], n_r['numerator']) < MIN_EVENTS: continue
        r = irr_ci(n_c['numerator'], n_c['denominator'],
                   n_r['numerator'], n_r['denominator'])
        if not np.isfinite(r['irr']): continue
        rank_score = abs(r['log_irr']) - 1.96 * r['se_log_irr']
        candidates.append({
            'sequence': keys[0],
            'third_idx': keys[1],
            'third_name': keys[2],
            'stratification': strat,
            'irr': r['irr'], 'rank_score': rank_score,
        })
cand = pd.DataFrame(candidates).sort_values('rank_score', ascending=False)
top60 = cand.head(60)
print(f'Computed {len(cand)} deprivation contrasts; top-60 retained.')

In [ ]:
# Now bin the top-60 third conditions into the four endpoint groups
def endpoint_of(third_name):
    if third_name == 'COPD':         return 'COPD'
    if third_name == 'DRUG_ALCOHOL': return 'DA'
    if third_name == 'HF':           return 'HF'
    if third_name == 'DEPRESSION':   return 'DEP'
    return 'OTHER'

top60['endpoint'] = top60['third_name'].apply(endpoint_of)
panel_B_counts = top60['endpoint'].value_counts().reindex(
    ['DA', 'COPD', 'DEP', 'HF', 'OTHER'], fill_value=0)
panel_B_counts

## Render the figure

In [ ]:
COL = {'COPD': '#185FA5', 'DA': '#C04828', 'HF': '#0F6E56', 'DEP': '#7F4FA0',
       'OTHER': '#888780', 'text': '#1F1F1E', 'mute': '#666660'}
LABELS = {'COPD': 'COPD', 'DA': 'Drug/alcohol misuse', 'HF': 'Heart failure',
          'DEP': 'Depression', 'OTHER': 'Other endpoints'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.edgecolor': '#999992', 'axes.linewidth': 0.6})

fig = plt.figure(figsize=(13.5, 10.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[2.4, 1.0], wspace=0.35,
                       left=0.11, right=0.97, top=0.86, bottom=0.18)
axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])

# === Panel A: grouped lollipop ===
n = len(panel_A)
axA.set_xscale('log'); axA.set_xlim(0.9, 8); axA.set_ylim(-0.6, n - 0.4)

# group bands
groups = {}
for i, r in panel_A.iterrows():
    groups.setdefault(r['endpoint'], []).append(i)
for ep, idxs in groups.items():
    axA.axhspan(min(idxs) - 0.5, max(idxs) + 0.5,
                color=COL[ep], alpha=0.07, zorder=0)

axA.axvline(1.0, color='#999992', lw=1.0, ls='--', zorder=1)
axA.text(1.0, -0.5, 'IRR = 1', fontsize=7.6, color=COL['mute'],
         ha='center', va='top')

for i, r in panel_A.iterrows():
    y = n - 1 - i
    c = COL[r['endpoint']]
    axA.plot([r['lo'], r['hi']], [y, y], color=c, lw=2.2,
             solid_capstyle='round', zorder=3)
    axA.plot([r['irr']], [y], 'o', color=c, ms=8.5, mec='white', mew=1.2,
             zorder=4)
    axA.text(1.05, y + 0.32, r['label'], fontsize=9.2, fontweight='bold')
    axA.text(1.05, y + 0.08, r['sublabel'], fontsize=7.7, style='italic',
             color=COL['mute'])
    axA.text(7.7, y + 0.20,
             f"IRR = {r['irr']:.2f}  [{r['lo']:.2f}-{r['hi']:.2f}]",
             fontsize=8.7, fontweight='bold', ha='right')
    axA.text(7.7, y - 0.22, f"N = {r['n_comp']:,} vs {r['n_ref']:,} events",
             fontsize=7.6, color=COL['mute'], ha='right')

# endpoint brackets outside left edge
for ep, idxs in groups.items():
    y_lo, y_hi = axA.get_ylim()
    ymin = (n - 1 - max(idxs) - 0.4 - y_lo) / (y_hi - y_lo)
    ymax = (n - 1 - min(idxs) + 0.4 - y_lo) / (y_hi - y_lo)
    axA.plot([-0.025, -0.025], [ymin, ymax],
             color=COL[ep], lw=4, solid_capstyle='butt',
             clip_on=False, transform=axA.transAxes)
    axA.text(-0.037, (ymin + ymax) / 2, f'→ {LABELS[ep]}',
             fontsize=9, fontweight='bold', color=COL[ep],
             ha='right', va='center', rotation=90, clip_on=False,
             transform=axA.transAxes)

axA.set_xticks([1, 2, 3, 4, 5, 6, 8])
axA.set_xticklabels(['1', '2', '3', '4', '5', '6', '8'])
axA.set_xlabel('Crude incidence rate ratio (log scale)\n'
               'most-deprived vs least-deprived, contrast pair given per row',
               fontsize=9.5)
axA.set_yticks([])
for sp in ('left', 'top', 'right'): axA.spines[sp].set_visible(False)
axA.text(0, 1.085, 'A', transform=axA.transAxes,
         fontsize=18, fontweight='bold')
axA.text(0.04, 1.085,
         'Selected length-three deprivation contrasts, grouped by endpoint',
         transform=axA.transAxes, fontsize=12.5, fontweight='bold')

# === Panel B: endpoint concentration ===
y_pos = np.arange(len(panel_B_counts))[::-1]
total = panel_B_counts.sum()
for y, (k, c) in zip(y_pos, panel_B_counts.items()):
    pct = c / total * 100
    axB.barh(y, pct, color=COL[k], height=0.7,
             edgecolor='white', linewidth=0.8)
    axB.text(-2.5, y, LABELS[k], fontsize=9, ha='right', va='center')
    axB.text(pct + 1.5, y, f'{c}/{total} ({pct:.0f}%)',
             fontsize=8.4, va='center')
axB.set_xlim(0, max(panel_B_counts.values) / total * 100 * 1.35)
axB.set_ylim(-0.6, len(panel_B_counts) - 0.2)
axB.set_yticks([]); axB.set_xlabel('% of top-60 length-3 deprivation candidates',
                                   fontsize=9.5)
for sp in ('left', 'top', 'right'): axB.spines[sp].set_visible(False)
axB.text(0, 1.085, 'B', transform=axB.transAxes,
         fontsize=18, fontweight='bold')
axB.text(0.07, 1.085, 'Third-condition concentration\nin top-60 candidates',
         transform=axB.transAxes, fontsize=12.5, fontweight='bold',
         linespacing=1.15)

fig.suptitle('Length-three deprivation contrasts converge on respiratory, '
             'substance-use, cardiovascular and mental-health endpoints',
             fontsize=13.0, fontweight='bold', y=0.965)
plt.show()

## What this notebook proved

Both panels of Figure 2 were re-derived from the parquet deposit in this notebook. Panel A's IRRs were re-computed from `get_sequence` calls; Panel B's convergence claim was re-derived by re-running the candidate ranking on the length-3 estimates.

If you want to explore further:

- Run `curate_ordered_transition_candidates.py --sequence-length 3 --min-events 30 --top-n 60` and inspect the `concentration_report_L3.txt` file it writes.
- Change the four endpoint groups in `endpoint_of()` above to see how the convergence story shifts with a different bucketing.
- Lower `MIN_EVENTS` to 10 to see the sensitivity-threshold version of the same figure.